# 02 - Baseline Models: Isolation Forest and Local Outlier Factor (LOF)

**Purpose:**
- Load processed features matrix.
- Apply `StandardScaler` to the 7 features (ensuring numerical parity, fixing BUG-05).
- Fit Isolation Forest and Local Outlier Factor using the authoritative 7-feature subset (fixing BUG-01).
- Unify column names and encode all anomaly labels using strict scikit-learn standard (`-1` = Anomaly, `1` = Normal) (fixing BUG-06 & BUG-07).
- Verify exact count parity with the accepted manuscript (e.g., exactly 5,000 IF anomalies).

**Inputs:**
- `data/processed/features_raw.parquet`

**Outputs:**
- `data/processed/anomaly_labels.parquet`
- `outputs/figures/fig4_if_scatter.png`
- `outputs/figures/fig5_lof_scatter.png`
- `outputs/tables/table1_anomaly_counts.csv`\n

In [ ]:
# Cell 01: Mount Storage & Bootstrap Paths
import os
import sys
from pathlib import Path

# Dual-Environment Parity Setup
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Paper1_Revision')
else:
    BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Bootstrap expected directories
for folder in ['data/raw', 'data/interim', 'data/processed', 
               'outputs/figures', 'outputs/tables', 'outputs/models', 
               'outputs/notebook_exports']:
    (BASE_DIR / folder).mkdir(parents=True, exist_ok=True)
    
print(f"Base directory set to: {BASE_DIR}")\n

In [ ]:
# Cell 02: Imports, Global Seeds & Style
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
import warnings

warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'figure.dpi': 300,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.autolayout': True
})\n

In [ ]:
# Cell 03: Load Processed Features
input_path = BASE_DIR / 'data' / 'processed' / 'features_raw.parquet'
print(f"Loading features from {input_path}...")
df_features = pd.read_parquet(input_path)

print(f"Data shape: {df_features.shape}")
assert df_features.shape == (100000, 7), "Expected exactly (100000, 7) from NB01"

# Extract feature names for later use
features = df_features.columns.tolist()\n

In [ ]:
# Cell 04: Standard Scaling
# NOTE (Fix for BUG-05): The original notebook applied StandardScaler BEFORE fitting Isolation Forest. 
# We replicate this exactly to guarantee numeric parity.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features)

print(f"Scaled feature matrix shape: {X_scaled.shape}")\n

In [ ]:
# Cell 05: Fit Isolation Forest
# Manuscript parameters: n_estimators=100, contamination=0.05, random_state=42
if_model = IsolationForest(
    n_estimators=100, 
    contamination=0.05, 
    random_state=42,
    n_jobs=-1
)

print("Fitting Isolation Forest...")
if_labels = if_model.fit_predict(X_scaled)

# Ensure scikit-learn standard (-1 = anomaly, 1 = normal)
if_anomaly_count = (if_labels == -1).sum()
print(f"Isolation Forest anomalies found: {if_anomaly_count}")

# PARITY ASSERTION: Must match Table 1 of the manuscript
assert if_anomaly_count == 5000, f"Parity failure: Expected exactly 5000 IF anomalies, got {if_anomaly_count}" \n

In [ ]:
# Cell 06: Fit Local Outlier Factor (LOF) - Auto & 0.05
# Manuscript parameters: n_neighbors=20
# Fix for BUG-01: Run on all 7 scaled features, NOT a 2-feature subset.
# Fix for BUG-02: Do NOT run contamination=0.01

print("Fitting LOF (contamination='auto')...")
lof_auto_model = LocalOutlierFactor(n_neighbors=20, contamination='auto', n_jobs=-1)
lof_auto_labels = lof_auto_model.fit_predict(X_scaled)

print("Fitting LOF (contamination=0.05)...")
lof_05_model = LocalOutlierFactor(n_neighbors=20, contamination=0.05, n_jobs=-1)
lof_05_labels = lof_05_model.fit_predict(X_scaled)

lof_auto_anomaly_count = (lof_auto_labels == -1).sum()
lof_05_anomaly_count = (lof_05_labels == -1).sum()

print(f"LOF (Auto) anomalies found: {lof_auto_anomaly_count}")
print(f"LOF (0.05) anomalies found: {lof_05_anomaly_count}")

# PARITY ASSERTION: For contamination=0.05, we mathematically expect exactly 5000 anomalies
assert lof_05_anomaly_count == 5000, f"Parity failure: Expected exactly 5000 LOF(0.05) anomalies, got {lof_05_anomaly_count}" \n

In [ ]:
# Cell 07: Construct Canonical Labels DataFrame
# Fix for BUG-06 & 07: Unify column names and strict -1/1 encodings
df_labels = pd.DataFrame({
    'IF_Label': if_labels,
    'LOF_Auto_Label': lof_auto_labels,
    'LOF_05_Label': lof_05_labels
})

# Verify no missing values and correct mapping
for col in df_labels.columns:
    unique_vals = df_labels[col].unique()
    assert set(unique_vals) == {1, -1}, f"Column {col} has invalid values: {unique_vals}"

# Create Table 1 summary
table1_data = {
    'Model': ['Isolation Forest', 'LOF (Auto)', 'LOF (0.05)'],
    'Contamination': [0.05, 'Auto', 0.05],
    'Normal Count (1)': [
        (df_labels['IF_Label'] == 1).sum(),
        (df_labels['LOF_Auto_Label'] == 1).sum(),
        (df_labels['LOF_05_Label'] == 1).sum()
    ],
    'Anomaly Count (-1)': [
        (df_labels['IF_Label'] == -1).sum(),
        (df_labels['LOF_Auto_Label'] == -1).sum(),
        (df_labels['LOF_05_Label'] == -1).sum()
    ]
}

df_table1 = pd.DataFrame(table1_data)
display(df_table1)

# Export Table 1
table1_path = BASE_DIR / 'outputs' / 'tables' / 'table1_anomaly_counts.csv'
df_table1.to_csv(table1_path, index=False)
print(f"Exported Table 1 to {table1_path}")\n

In [ ]:
# Cell 08: Visualizations - Scatter Plots
# We plot the anomaly distribution against two illustrative dimensions: 
# 'Number of Services' vs 'Average Medicare Payment Amount'

x_col = 'Number of Services'
y_col = 'Average Medicare Payment Amount'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Isolation Forest
sns.scatterplot(
    x=df_features[x_col], 
    y=df_features[y_col], 
    hue=df_labels['IF_Label'].map({1: 'Normal', -1: 'Anomaly'}),
    palette={'Normal': '#1f77b4', 'Anomaly': '#d62728'},
    alpha=0.6,
    s=15,
    ax=axes[0]
)
axes[0].set_title('Isolation Forest Anomalies', fontsize=14)
axes[0].set_xlabel(x_col)
axes[0].set_ylabel(y_col)

# Plot 2: LOF (0.05)
sns.scatterplot(
    x=df_features[x_col], 
    y=df_features[y_col], 
    hue=df_labels['LOF_05_Label'].map({1: 'Normal', -1: 'Anomaly'}),
    palette={'Normal': '#1f77b4', 'Anomaly': '#d62728'},
    alpha=0.6,
    s=15,
    ax=axes[1]
)
axes[1].set_title('Local Outlier Factor (0.05) Anomalies', fontsize=14)
axes[1].set_xlabel(x_col)
axes[1].set_ylabel(y_col)

plt.tight_layout()

# Export figures
fig4_path = BASE_DIR / 'outputs' / 'figures' / 'fig4_if_scatter.png'
fig5_path = BASE_DIR / 'outputs' / 'figures' / 'fig5_lof_scatter.png'

# Save them individually as required by the PRD
fig_if = axes[0].get_figure()
fig_if.savefig(fig4_path, dpi=300, bbox_inches='tight')

fig_lof = axes[1].get_figure()
fig_lof.savefig(fig5_path, dpi=300, bbox_inches='tight')

print(f"Exported Figure 4 to {fig4_path}")
print(f"Exported Figure 5 to {fig5_path}")
plt.show()\n

In [ ]:
# Cell 09: Export Labels Matrix
# This parquet file acts as the deterministic input for NB04, NB05, and NB06
export_path = BASE_DIR / 'data' / 'processed' / 'anomaly_labels.parquet'
df_labels.to_parquet(export_path, index=False)

print(f"Successfully exported {df_labels.shape} matrix to {export_path}")\n